In [5]:
import cv2
import mediapipe as mp
from keras.models import load_model
import numpy as np

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils


In [6]:
model = load_model("E:\\ML\\handsign\\models\\model_handsign2.h5") 
danh_sach_chu = ["a", "b", "c", "d", "e", "f", "g", "h", "i", "k", "l", "m", "n", "o", "p", "q", "r", "s", "t", "u", "v", "w", "x", "y"]


In [7]:
def khoang_cach_toa_do(hand_lm):
    danh_sach_toa_do = []
    for lm in hand_lm.landmark:
        danh_sach_toa_do.append([lm.x, lm.y, lm.z])
    toa_do = np.array(danh_sach_toa_do)
    co_tay = toa_do[0]
    chuan_hoa = (toa_do - co_tay)
    return chuan_hoa.flatten()


In [8]:
cam = cv2.VideoCapture(0)
while True:
    ret, frame = cam.read()    
    if not ret: break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)    
    h_frames, w_frames, c = frame.shape

    if results.multi_hand_landmarks:
        for hand_lm in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_lm, mp_hands.HAND_CONNECTIONS)
            
            du_lieu = np.array([khoang_cach_toa_do(hand_lm)])
            du_doan = model.predict(du_lieu, verbose=0)   
            phan_tram = np.max(du_doan) 
            ket_qua_chu = danh_sach_chu[np.argmax(du_doan)]
            
            if phan_tram > 0.6:
                x_co_tay = int(hand_lm.landmark[0].x * w_frames)
                y_co_tay = int(hand_lm.landmark[0].y * h_frames)
                cv2.putText(frame, f"{ket_qua_chu.upper()}: {phan_tram*100:.1f}%", 
                            (x_co_tay - 20, y_co_tay - 20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Test Handsign", frame)
    if cv2.waitKey(1) & 0xFF == ord('l'): break
cam.release()
cv2.destroyAllWindows()
